<a href="https://colab.research.google.com/github/afzal/WinYourDay/blob/main/TODO_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.2 MB/s eta 0:00:00


In [4]:
# --- Colab setup (ReportLab for PDF) ---
!pip -q install reportlab

import os, uuid, tempfile
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

# Optional: in Colab, this can help if widget rendering is flaky in your session.
# (Mostly needed for non-default widgets, but harmless to try.)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

# =========================
# Config + schema
# =========================
PLAN_FILE = "tomorrow_plan.csv"
SESSION_ID = uuid.uuid4().hex[:8]

COLS = ["Task", "Category", "Start", "End", "Status", "Worked_Minutes"]
STATUS_OPTIONS = ["Not Checked", "Done", "Partial", "Missed"]

# Single source of truth during a session:
plan_df = pd.DataFrame(columns=COLS)

# UI redraw lock (prevents re-entrant callbacks during refresh)
REFRESHING = False

# Summary cache for PDF export
summary_cache = {
    "computed": False,
    "total_planned": 0,
    "total_worked": 0,
    "total_missed": 0,
    "total_saved": 0,
    "completion_rate": 0.0,
    "title": "",
    "msg": "",
    "charts": (None, None, None),
}

# =========================
# Persistence helpers
# =========================
def _empty_plan():
    return pd.DataFrame(columns=COLS)

def _coerce_plan(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure required columns exist, valid dtypes, valid statuses, and RangeIndex."""
    df = df.copy()

    for c in COLS:
        if c not in df.columns:
            df[c] = "" if c != "Worked_Minutes" else 0

    # Normalize textual columns
    df["Task"] = df["Task"].fillna("").astype(str)
    df["Category"] = df["Category"].fillna("").astype(str)
    df["Start"] = df["Start"].fillna("00:00").astype(str)
    df["End"] = df["End"].fillna("00:00").astype(str)

    # Normalize status
    df["Status"] = df["Status"].fillna("Not Checked").astype(str)
    df.loc[~df["Status"].isin(STATUS_OPTIONS), "Status"] = "Not Checked"

    # Normalize worked minutes (int)
    df["Worked_Minutes"] = pd.to_numeric(df["Worked_Minutes"], errors="coerce").fillna(0).astype(int)

    # Clean index
    df = df.reset_index(drop=True)
    return df

def load_plan():
    """Load once at startup (or via Reload button). Do NOT call during UI events."""
    global plan_df
    if os.path.exists(PLAN_FILE):
        df = pd.read_csv(PLAN_FILE)
        plan_df = _coerce_plan(df)
    else:
        plan_df = _empty_plan()

def save_plan():
    """Save immediately after each edit. Atomic replace to avoid partial writes."""
    global plan_df
    plan_df = _coerce_plan(plan_df)

    tmp_path = PLAN_FILE + ".tmp"
    plan_df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, PLAN_FILE)

# =========================
# Time + calculated fields
# =========================
def time_to_minutes(start: str, end: str) -> int:
    fmt = "%H:%M"
    s = datetime.strptime(start, fmt)
    e = datetime.strptime(end, fmt)
    diff = (e - s).total_seconds() / 60
    if diff < 0:
        diff += 24 * 60
    return int(diff)

def worked_options_for(planned: int, step: int = 5):
    if planned <= 0:
        return [0]
    opts = list(range(0, planned + 1, step))
    if planned not in opts:
        opts.append(planned)
    return sorted(set(opts))

def add_calculated(df: pd.DataFrame) -> pd.DataFrame:
    """Adds Planned_Minutes, Missed_Minutes, Saved_Minutes, and safe Worked_Actual."""
    df = _coerce_plan(df)

    if len(df) == 0:
        return df.assign(Planned_Minutes=[], Worked_Actual=[], Missed_Minutes=[], Saved_Minutes=[])

    planned = df.apply(lambda r: time_to_minutes(r["Start"], r["End"]), axis=1).astype(int)
    df["Planned_Minutes"] = planned

    worked_actual = []
    missed = []
    saved = []

    for _, r in df.iterrows():
        p = int(r["Planned_Minutes"])
        s = str(r["Status"])
        w = int(r["Worked_Minutes"])
        w = max(0, min(w, p))

        if s == "Missed":
            w2 = 0
            m2 = p
            sv2 = 0
        elif s == "Not Checked":
            w2 = 0
            m2 = p
            sv2 = 0
        elif s == "Partial":
            w2 = w
            m2 = max(p - w2, 0)
            sv2 = 0
        elif s == "Done":
            # Done counts as complete even if finished early
            # (Saved time is not "missed")
            w2 = w if w > 0 else p
            m2 = 0
            sv2 = max(p - w2, 0)
        else:
            w2 = 0
            m2 = p
            sv2 = 0

        worked_actual.append(w2)
        missed.append(m2)
        saved.append(sv2)

    df["Worked_Actual"] = worked_actual
    df["Missed_Minutes"] = missed
    df["Saved_Minutes"] = saved
    return df

# =========================
# PDF export
# =========================
def export_pdf():
    if not summary_cache["computed"]:
        with message_out:
            message_out.clear_output(wait=True)
            print("Run Summary first, then Save PDF.")
        return

    today = datetime.now().strftime("%Y-%m-%d")
    total_worked = summary_cache["total_worked"]
    filename = f"Daily_Report_{today}_{total_worked/60:.2f}h.pdf"

    styles = getSampleStyleSheet()

    # Keep PDF text ASCII-safe (emojis often won't render with default fonts)
    title = summary_cache["title"].encode("ascii", "ignore").decode()
    msg = summary_cache["msg"].encode("ascii", "ignore").decode()

    doc = SimpleDocTemplate(filename, pagesize=letter)
    content = []

    content.append(Paragraph(f"<b>{title}</b>", styles["Title"]))
    content.append(Spacer(1, 12))
    content.append(Paragraph(f"<b>Date:</b> {today}", styles["Normal"]))
    content.append(Spacer(1, 12))

    content.append(Paragraph(f"<b>Total Planned:</b> {summary_cache['total_planned']/60:.2f} hours", styles["Normal"]))
    content.append(Paragraph(f"<b>Total Worked:</b> {summary_cache['total_worked']/60:.2f} hours", styles["Normal"]))
    content.append(Paragraph(f"<b>Total Missed:</b> {summary_cache['total_missed']/60:.2f} hours", styles["Normal"]))
    content.append(Paragraph(f"<b>Total Saved (finished early):</b> {summary_cache['total_saved']/60:.2f} hours", styles["Normal"]))
    content.append(Paragraph(f"<b>Completion Rate:</b> {summary_cache['completion_rate']:.1f}%", styles["Normal"]))
    content.append(Spacer(1, 12))

    content.append(Paragraph(f"<b>Feedback:</b> {msg}", styles["Normal"]))
    content.append(Spacer(1, 16))

    for path in summary_cache["charts"]:
        if path and os.path.exists(path):
            content.append(RLImage(path, width=500, height=220))
            content.append(Spacer(1, 12))

    doc.build(content)

    with message_out:
        message_out.clear_output(wait=True)
        print(f"PDF saved: {filename}")

# =========================
# Widgets (top entry)
# =========================
category_options = ["Research", "Job Search", "Teaching", "Health", "Personal"]

task_input = widgets.Text(placeholder="Task", layout=widgets.Layout(width="200px"))
category_input = widgets.Dropdown(options=category_options, value="Research", layout=widgets.Layout(width="170px"))

hours = [f"{i:02d}" for i in range(24)]
minutes = [f"{i:02d}" for i in range(0, 60, 5)]

start_h = widgets.Dropdown(options=hours, value="09", layout=widgets.Layout(width="70px"))
start_m = widgets.Dropdown(options=minutes, value="00", layout=widgets.Layout(width="70px"))
end_h = widgets.Dropdown(options=hours, value="10", layout=widgets.Layout(width="70px"))
end_m = widgets.Dropdown(options=minutes, value="00", layout=widgets.Layout(width="70px"))

add_btn = widgets.Button(description="Add", button_style="success", layout=widgets.Layout(width="90px"))
summary_btn = widgets.Button(description="Summary", button_style="info")
pdf_btn = widgets.Button(description="Save PDF", button_style="success", disabled=True)
clear_btn = widgets.Button(description="Clear All", button_style="danger")
reload_btn = widgets.Button(description="Reload CSV", button_style="warning")

message_out = widgets.Output()
table_out = widgets.Output()
summary_out = widgets.Output()

def intervals_overlap(start1, end1, start2, end2):
    s1 = datetime.strptime(start1, "%H:%M")
    e1 = datetime.strptime(end1, "%H:%M")
    s2 = datetime.strptime(start2, "%H:%M")
    e2 = datetime.strptime(end2, "%H:%M")

    # only same-day intervals supported here
    return s1 < e2 and s2 < e1
# =========================
# Core actions
# =========================
def add_task(_):
    global plan_df

    task = task_input.value.strip()
    category = category_input.value
    start = f"{start_h.value}:{start_m.value}"
    end = f"{end_h.value}:{end_m.value}"

    with message_out:
        clear_output()

        if not task:
            print("Enter a task name.")
            return

        if start == end:
            print("Start and end time cannot be the same.")
            return

        # prevent backwards time like 15:00 to 14:00
        if datetime.strptime(start, "%H:%M") >= datetime.strptime(end, "%H:%M"):
            print("End time must be later than start time.")
            return

        # overlap check
        for _, row in plan_df.iterrows():
            existing_start = str(row["Start"])
            existing_end = str(row["End"])

            if intervals_overlap(start, end, existing_start, existing_end):
                print(f"Time overlaps with existing task: {row['Task']} ({existing_start}-{existing_end})")
                return

        new_row = {
            "Task": task,
            "Category": category,
            "Start": start,
            "End": end,
            "Status": "Not Checked",
            "Worked_Minutes": 0
        }

        plan_df.loc[len(plan_df)] = new_row
        save_plan()
        print(f"Task added: {task}")

    task_input.value = ""
    refresh_table()

def delete_task(i: int):
    global plan_df
    if 0 <= i < len(plan_df):
        plan_df = plan_df.drop(index=i).reset_index(drop=True)
        save_plan()
    refresh_table()

def clear_all(_):
    global plan_df
    plan_df = _empty_plan()
    save_plan()
    pdf_btn.disabled = True
    summary_cache["computed"] = False

    with summary_out:
        summary_out.clear_output(wait=True)
    with message_out:
        message_out.clear_output(wait=True)
        print("All tasks cleared.")

    refresh_table()

def reload_from_disk(_):
    load_plan()
    with message_out:
        message_out.clear_output(wait=True)
        print("Reloaded from CSV.")
    refresh_table()

# =========================
# Table rendering (stable)
# =========================
def refresh_table():
    global REFRESHING
    REFRESHING = True
    try:
        table_out.clear_output(wait=True)
        with table_out:
            if len(plan_df) == 0:
                print("No tasks yet.")
                return

            dfc = add_calculated(plan_df)

            # Header
            header = widgets.HBox([
                widgets.HTML("<b style='width:110px; display:inline-block;'>Task</b>"),
                widgets.HTML("<b style='width:120px; display:inline-block;'>Category</b>"),
                widgets.HTML("<b style='width:70px; display:inline-block;'>Start</b>"),
                widgets.HTML("<b style='width:70px; display:inline-block;'>End</b>"),
                widgets.HTML("<b style='width:115px; display:inline-block;'>Status</b>"),
                widgets.HTML("<b style='width:95px; display:inline-block;'>Worked</b>"),
                widgets.HTML("<b style='width:85px; display:inline-block;'>Planned</b>"),
                widgets.HTML("<b style='width:85px; display:inline-block;'>Missed</b>"),
                widgets.HTML("<b style='width:85px; display:inline-block;'>Saved</b>"),
                widgets.HTML("<b style='width:220px; display:inline-block;'>Reward</b>"),
                widgets.HTML("<b style='width:60px; display:inline-block;'>Del</b>"),
            ])
            display(header)

            for i, row in dfc.iterrows():
                planned = int(row["Planned_Minutes"])
                status = str(plan_df.loc[i, "Status"])
                worked = int(plan_df.loc[i, "Worked_Minutes"])
                missed = int(row["Missed_Minutes"])
                saved = int(row["Saved_Minutes"])

                # Ensure worked value is a valid choice
                worked_opts = worked_options_for(planned)
                if worked not in worked_opts:
                    worked_opts = sorted(set(worked_opts + [worked]))

                status_dd = widgets.Dropdown(
                    options=STATUS_OPTIONS,
                    value=status,
                    layout=widgets.Layout(width="115px")
                )

                worked_dd = widgets.Dropdown(
                    options=worked_opts,
                    value=worked,
                    layout=widgets.Layout(width="90px")
                )

                # Worked is only editable for Done/Partial
                worked_dd.disabled = status not in ["Done", "Partial"]

                reward = widgets.HTML(layout=widgets.Layout(width="220px"))
                if status == "Done" and saved > 0:
                    reward.value = f"<span style='color:green; font-weight:700;'>👏 Finished {saved} min early!</span>"
                elif status == "Done":
                    reward.value = "<span style='color:green; font-weight:700;'>🏆 Completed!</span>"
                elif status == "Partial" and worked > 0:
                    reward.value = "<span style='color:orange; font-weight:700;'>💪 Progress!</span>"
                elif status == "Missed":
                    reward.value = "<span style='color:red; font-weight:700;'>😴 Missed</span>"
                else:
                    reward.value = "<span style='color:gray;'>—</span>"

                del_btn = widgets.Button(description="Del", button_style="danger",
                                        layout=widgets.Layout(width="55px"))

                # Safe handlers: ignore during refresh, check index bounds, only respond to value changes
                def on_status_change(change, idx=i):
                    global plan_df, REFRESHING
                    if REFRESHING or change.get("name") != "value":
                        return
                    if not (0 <= idx < len(plan_df)):
                        return

                    new_status = change["new"]
                    plan_df.loc[idx, "Status"] = new_status

                    p = time_to_minutes(str(plan_df.loc[idx, "Start"]), str(plan_df.loc[idx, "End"]))
                    w = int(plan_df.loc[idx, "Worked_Minutes"])

                    if new_status == "Done":
                        # Default worked to planned if unset; user can then reduce it for "finished early"
                        if w <= 0:
                            plan_df.loc[idx, "Worked_Minutes"] = p
                        else:
                            plan_df.loc[idx, "Worked_Minutes"] = max(0, min(w, p))
                    elif new_status == "Partial":
                        plan_df.loc[idx, "Worked_Minutes"] = max(0, min(w, p))
                    else:  # Missed or Not Checked
                        plan_df.loc[idx, "Worked_Minutes"] = 0

                    save_plan()
                    refresh_table()

                def on_worked_change(change, idx=i):
                    global plan_df, REFRESHING
                    if REFRESHING or change.get("name") != "value":
                        return
                    if not (0 <= idx < len(plan_df)):
                        return

                    # Only accept worked edits if status allows it
                    st = str(plan_df.loc[idx, "Status"])
                    if st not in ["Done", "Partial"]:
                        return

                    p = time_to_minutes(str(plan_df.loc[idx, "Start"]), str(plan_df.loc[idx, "End"]))
                    v = int(change["new"])
                    v = max(0, min(v, p))
                    plan_df.loc[idx, "Worked_Minutes"] = v

                    save_plan()
                    refresh_table()

                status_dd.observe(on_status_change, names="value")
                worked_dd.observe(on_worked_change, names="value")

                def on_delete_clicked(_, idx=i):
                    delete_task(idx)

                del_btn.on_click(on_delete_clicked)

                row_widget = widgets.HBox([
                    widgets.Label(str(row["Task"]), layout=widgets.Layout(width="110px")),
                    widgets.Label(str(row["Category"]), layout=widgets.Layout(width="120px")),
                    widgets.Label(str(row["Start"]), layout=widgets.Layout(width="70px")),
                    widgets.Label(str(row["End"]), layout=widgets.Layout(width="70px")),
                    status_dd,
                    worked_dd,
                    widgets.Label(str(planned), layout=widgets.Layout(width="85px")),
                    widgets.Label(str(missed), layout=widgets.Layout(width="85px")),
                    widgets.Label(str(saved), layout=widgets.Layout(width="85px")),
                    reward,
                    del_btn
                ])
                display(row_widget)

    finally:
        REFRESHING = False

# =========================
# Summary + charts + PDF enable
# =========================
def show_summary(_):
    global summary_cache

    summary_out.clear_output(wait=True)
    with summary_out:
        dfc = add_calculated(plan_df)
        if len(dfc) == 0:
            print("No data.")
            return

        total_planned = int(dfc["Planned_Minutes"].sum())
        total_worked = int(dfc["Worked_Actual"].sum())
        total_missed = int(dfc["Missed_Minutes"].sum())
        total_saved = int(dfc["Saved_Minutes"].sum())

        # Completion should treat Done as complete even if early:
        completion_rate = ((total_planned - total_missed) / total_planned * 100) if total_planned else 0.0

        # Feedback
        if completion_rate >= 90:
            msg = "🔥 Outstanding. You’re crushing it."
            title = "🏆 Elite Day"
        elif completion_rate >= 75:
            msg = "👏 Great work. Strong consistency."
            title = "🚀 Productive Day"
        elif completion_rate >= 50:
            msg = "💪 You moved forward. Keep building."
            title = "🙂 Solid Day"
        elif completion_rate >= 25:
            msg = "🙂 Some progress. Tomorrow is your comeback."
            title = "🐢 Slow Day"
        else:
            msg = "😅 Fresh start tomorrow. Small wins count."
            title = "🧊 Ice Day"

        # Show summary cards
        display(HTML(f"""
        <div style="padding:10px; border-radius:10px; background:#f3f3f3;">
          <div style="font-size:18px;"><b>{title}</b></div>
          <div style="margin-top:6px;">{msg}</div>
          <div style="margin-top:10px; display:flex; gap:14px; flex-wrap:wrap;">
            <div>📅 Planned: <b>{total_planned/60:.2f}h</b></div>
            <div>⚡ Worked: <b>{total_worked/60:.2f}h</b></div>
            <div>😴 Missed: <b>{total_missed/60:.2f}h</b></div>
            <div>🏁 Saved: <b>{total_saved/60:.2f}h</b></div>
            <div>📊 Completion: <b>{completion_rate:.1f}%</b></div>
          </div>
        </div>
        """))

        # Charts
        today = datetime.now().strftime("%Y-%m-%d")
        chart1 = f"planned_vs_worked_{today}.png"
        chart2 = f"completion_{today}.png"
        chart3 = f"day_view_{today}.png"

        # Chart 1: planned vs worked by task
        plt.figure(figsize=(8, 4))
        x = range(len(dfc))
        plt.bar(x, dfc["Planned_Minutes"], label="Planned")
        plt.bar(x, dfc["Worked_Actual"], label="Worked")
        plt.xticks(x, dfc["Task"], rotation=45, ha="right")
        plt.ylabel("Minutes")
        plt.title("Planned vs Worked (per task)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(chart1, bbox_inches="tight")
        plt.show()
        plt.close()

        # Chart 2: completion rate
        plt.figure(figsize=(8, 2))
        plt.barh(["Completion"], [completion_rate])
        plt.xlim(0, 100)
        plt.xlabel("Percent")
        plt.title("Completion % (Done counts as complete)")
        plt.tight_layout()
        plt.savefig(chart2, bbox_inches="tight")
        plt.show()
        plt.close()

        # Chart 3: 24h view split
        total_day = 24 * 60
        other = max(total_day - total_planned, 0)

        plt.figure(figsize=(10, 2))
        left = 0
        plt.barh(["Day"], [total_worked], left=left, label="Worked")
        left += total_worked
        plt.barh(["Day"], [total_missed], left=left, label="Planned but missed")
        left += total_missed
        plt.barh(["Day"], [total_saved], left=left, label="Saved (finished early)")
        left += total_saved
        plt.barh(["Day"], [other], left=left, label="Other time")
        plt.xlim(0, total_day)
        plt.xlabel("Minutes")
        plt.title("24-hour view")
        plt.legend()
        plt.tight_layout()
        plt.savefig(chart3, bbox_inches="tight")
        plt.show()
        plt.close()

        # Cache for PDF
        summary_cache = {
            "computed": True,
            "total_planned": total_planned,
            "total_worked": total_worked,
            "total_missed": total_missed,
            "total_saved": total_saved,
            "completion_rate": float(completion_rate),
            "title": title,
            "msg": msg,
            "charts": (chart1, chart2, chart3),
        }

        pdf_btn.disabled = False

# =========================
# Wire buttons + build UI
# =========================
add_btn.on_click(add_task)
summary_btn.on_click(show_summary)
pdf_btn.on_click(lambda _: export_pdf())
clear_btn.on_click(clear_all)
reload_btn.on_click(reload_from_disk)

load_plan()

display(HTML(f"""
<h2>Tomorrow Plan</h2>
<div style="color: gray; font-size: 12px;">Session: {SESSION_ID} (run this cell once; avoid using older UIs from previous runs)</div>
"""))

display(widgets.HBox([
    widgets.HTML("<b style='width:200px;'>Task</b>"),
    widgets.HTML("<b style='width:170px;'>Category</b>"),
    widgets.HTML("<b style='width:140px;'>Start</b>"),
    widgets.HTML("<b style='width:140px;'>End</b>"),
    widgets.HTML("<b style='width:90px;'>Add</b>")
]))

display(widgets.HBox([
    task_input,
    category_input,
    widgets.HBox([start_h, start_m], layout=widgets.Layout(width="140px")),
    widgets.HBox([end_h, end_m], layout=widgets.Layout(width="140px")),
    add_btn
]))

display(message_out)

display(HTML("<h3>Planned Tasks</h3>"))
display(table_out)

display(widgets.HBox([summary_btn, pdf_btn, clear_btn, reload_btn]))
display(HTML("<h3>Summary + Visualizations</h3>"))
display(summary_out)

refresh_table()


Output()

Output()

Output()